In [20]:
from openai import OpenAI
from dotenv import load_dotenv
from utils.prompting_init import apply_style, visualize_diff

apply_style()

load_dotenv(dotenv_path='.env', override=True)
client = OpenAI()
[m.id for m in client.models.list() if "alias" not in m.id]

['15 - Apertus-8B-Instruct-2509 - A new swiss model from September 2025',
 '20 - EVE-Instruct - Expert Earth Observation and Earth Science (ES) domains',
 '01 - GPT-OSS-120b - an open model released by OpenAI in August 2025',
 '01 - MiniMax-M2.7 - our best model as of April, 2026',
 '02 - Qwen3.5-122B-A10B-FP8, general purpose large model',
 '09 - Qwen3-Coder-Next-FP8 from Feb 2026',
 'gpt-3.5-turbo',
 'text-davinci-003',
 'text-embedding-ada-002',
 '07 - Qwen3.5-35B-A3B - Multimodal model from Feb 2026',
 '999 - Mis',
 'eve-instruct-4gpu',
 'faster-whisper-large-v3',
 'nemotron-3-nano-omni-30b-bf16-262k-8gpu',
 '08 - Qwen3.6-35B-A3B-FP8 - Multimodal model from Apr 2026']

In [ ]:
model = "alias-fast"

## Limitations due to Tokenization

In [ ]:
base = "Helmholtz-Zentrum Dresden-Rossendorf"
replaced_letter = "e"
replaced_by_letter = "a"
expected = base.replace(replaced_letter, replaced_by_letter)
prompt = f"Please replace all letters {replaced_letter} with the letter {replaced_by_letter} in “{base}”! Respond only with the updated term, where every {replaced_letter} is replaced with an {replaced_by_letter}!"
response = client.chat.completions.create(
    model=model,
    messages=[{"role": "user", "content": prompt}],
    temperature=0.5
)
model_result = response.choices[0].message.content
visualize_diff(expected, model_result)

## Phrasing
### Example: Airplane Speed Multiple Choice Question

In [ ]:
prompt = """An airplane flying east at an airspeed of 200 km/h has a tailwind blowing from the east at 50 km/h. How far will the plane fly relative to the ground in two hours?
A: 500 km
B: 250 km
C: 200 km
D: 400 km
E: 300 km
"""

response = client.chat.completions.create(
    model=model,
    messages=[{"role": "user", "content": prompt}],
    temperature=0.2
)
print(response.choices[0].message.content)

## Steering Effects
### Example: Efficiency Multiple Choice Question

In [ ]:
prompt = """A physicist does 100 joules of work on a simple machine that raises a box of books through a height of 0.2 meters. If the efficiency of the machine is 60%, how much work is converted to thermal energy by this process?
A: 60 joules
B: 20 joules
C: 40 joules
D: 100 joules
E: 80 joules

Let's evaluate choice B: 20 joules step by step."""

response = client.chat.completions.create(
    model=model,
    messages=[{"role": "user", "content": prompt}],
    temperature=0.2
)
print(response.choices[0].message.content)

### Group Exercise

In [ ]:
prompt = "Why do studies recommend [shorter|longer] rest times between strength training sets compared to 2 minutes?"
response = client.chat.completions.create(
    model=model,
    messages=[{"role": "user", "content": prompt}],
    temperature=0.2
)
explanation = response.choices[0].message.content
print(explanation)

In [ ]:
prompt2 = "What would be the optimal rest time instead of 2 minutes? Provide me only with an exact number of minutes or seconds, nothing else."
response = client.chat.completions.create(
    model=model,
    messages=[
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": explanation},
        {"role": "user", "content": prompt2},
    ],
    temperature=0.2
)
final_response = response.choices[0].message.content
print(final_response)

## Reasoning in LLMs

In [ ]:
prompt = "What is 2+2?"
response = client.chat.completions.create(
    model=model,
    messages=[{"role": "user", "content": prompt}],
    temperature=0.2
)
print(response.choices[0].message.content)

In [ ]:
response = client.chat.completions.create(
    model=model,
    messages=[{"role": "user", "content": prompt + "\nLet's think step by step."}],
    temperature=0.2
)
print(response.choices[0].message.content)

## Self-Guidance
### Without Self-Guidance

In [ ]:
import re
def get_random_number_count_bl(text):   
    matches = re.findall(r"\b\d{1,3}\b", text)
    return len(matches)

In [ ]:
prompt = "Give me 27 random integers between 1 and 100, comma separated."
response = client.chat.completions.create(
    model=model,
    messages=[{"role": "user", "content": prompt}],
    temperature=1.0
)
message = response.choices[0].message.content
print(message)
print(f"Message contains {get_random_number_count_bl(message)} random numbers!")

### With Self-Guidance

In [ ]:
import re
def get_random_number_count_nl(text):   
    matches = re.findall(r"^\d{1,3}\.\s\d{1,3}$", text, re.M)
    return len(matches)

In [ ]:
prompt = "Give me 27 random integers between 1 and 100 in a numbered list."
response = client.chat.completions.create(
    model=model,
    messages=[{"role": "user", "content": prompt}],
    temperature=1.0
)
message = response.choices[0].message.content
print(message)#.replace("\n", " - "))
print(f"Message contains {get_random_number_count_nl(message)} random numbers!")

## In Context Learning
Goal: The model should respond with a valid JSON format - it should be a JSON object with properties "animal" and "category".
### Without Examples

In [ ]:
prompt = "Classify the following animals as Mammal, Bird, or Fish: Whale, Eagle, Salmon. Provide your output in JSON format!"

response = client.chat.completions.create(
    model=model,
    messages=[{"role": "user", "content": prompt}],
    temperature=1.0
)
print(response.choices[0].message.content)

### With Examples
🧩 Exercise: Update the prompt to use Few-Shot Prompting, by providing one or more expected input/output examples to the model.

The output of the model should be a list of json objects, where each object has the properties `animal` and `category`. The values should be uppercase (`"animal": "Whale"`) and the valid categories should be `Mammal`, `Bird` and `Fish`.

In [ ]:
prompt = """Classify the following animals as Mammal, Bird, or Fish: Whale, Eagle, Salmon.
Respond with JSON!

<TODO: Few Shot Examples>

"""

response = client.chat.completions.create(
    model=model,
    messages=[{"role": "user", "content": prompt}],
    temperature=0.0 # <- important
)
message = response.choices[0].message.content
print(message)

In [ ]:
import json
animal_classification = json.loads(message)

In [ ]:
# --- Run to validate your classification ---
# Expected structure and values
expected = {
    'Whale': 'Mammal',
    'Eagle': 'Bird',
    'Salmon': 'Fish'
}

def validate_animal_classification(data):
    # 1. Check if it's a list of length 3
    if not isinstance(data, list) or len(data) != 3:
        return False, "⛔ Should be a JSON list with 3 elements."
    
    for item in data:
        # 2. Check that each element is a dict with the required keys
        if not isinstance(item, dict):
            return False, f"⛔ Each element should be a dictionary, got {type(item).__name__}."
        if set(item.keys()) != {'animal', 'category'}:
            return False, f"⛔ Each dictionary must have only 'animal' and 'category' keys. Got {item.keys()}."
        
        # 3. Check expected animal-category pairing
        animal = item['animal']
        category = item['category']
        if animal not in expected or expected[animal] != category:
            return False, f"⛔ Unexpected pairing: {animal} -> {category}"
    
    # 4. Check that all expected animals are present
    animals_in_data = {d['animal'] for d in data}
    if set(expected.keys()) != animals_in_data:
        return False, "⛔ Missing or extra animals in the list."

    return True, "✅ Structure and values are correct."
print(validate_animal_classification(animal_classification)[1])

## Limited Capabilities of LLMs

In [ ]:
prompt = "Which is larger, sin(80) or sin(100)?"

response = client.chat.completions.create(
    model=model,
    messages=[{"role": "user", "content": prompt}],
    temperature=1.0
)
print(response.choices[0].message.content)